# OpenAI vs KURE-v1 임베딩 비교

전·월세 법령·판례 청크로 두 임베딩 모델을 같은 조건(1024차원·코사인)에서 비교한다.

- **OpenAI** `text-embedding-3-small` (`dimensions=1024` 로 축소)
- **KURE-v1** `nlpai-lab/KURE-v1` (한국어 검색 특화, 1024차원)

비교 항목: 임베딩 shape·소요 시간·처리량, 그리고 실제 법률 질문에 대한 top-k 검색 결과.
청킹은 `test/ingest_kure.py` 의 로직을 그대로 재사용한다.

## 0. 준비

```bash
# backend/ 에서
uv sync --group ingest          # openai · sentence-transformers · numpy 설치
```

- `.env` 에 `OPENAI_API_KEY` 필요 (이미 설정돼 있음).
- KURE-v1 최초 실행 시 모델(~2GB)을 HuggingFace 에서 내려받는다 (CPU 도 동작, GPU 면 빠름).
- OpenAI 임베딩은 유료 API 호출이다. 아래 `PER_FILE` 로 표본 크기를 줄여 비용을 통제한다.

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

# backend/ 를 sys.path 에 올려 test.ingest_kure(청킹 로직)를 재사용한다.
BACKEND = Path.cwd()
while not (BACKEND / 'test' / 'ingest_kure.py').exists():
    if BACKEND == BACKEND.parent:
        raise RuntimeError('backend/ 를 찾지 못했습니다. 노트북을 backend/tests/chunks 에서 여세요.')
    BACKEND = BACKEND.parent
sys.path.insert(0, str(BACKEND))
load_dotenv(BACKEND / '.env')

from test.ingest_kure import build_chunks, iter_records  # noqa: E402

DATA_DIR = BACKEND.parent / 'data' / '02_processed' / 'api_text'
print('backend :', BACKEND)
print('data    :', DATA_DIR, '(exists:', DATA_DIR.exists(), ')')
print('openai key:', 'OK' if os.getenv('OPENAI_API_KEY') else '없음')

## 1. 샘플 청크 로드

법령(`eflaw`)·판례(`prec_*`) 일부를 청킹한다. `PER_FILE` 을 키우면 표본이 커진다(비용·시간 증가).

In [ ]:
FILES = ['eflaw.jsonl', 'prec_전세사기.jsonl', 'prec_보증금권리.jsonl']
PER_FILE = 40            # 파일당 레코드 수 (비용·시간 절약용)
CHUNK_SIZE, OVERLAP = 1000, 150

chunks = []
for name in FILES:
    for rec in iter_records(DATA_DIR / name, limit=PER_FILE):
        chunks.extend(build_chunks(rec, CHUNK_SIZE, OVERLAP))

texts  = [c['embed_text'] for c in chunks]                 # 맥락 헤더 포함 (임베딩 입력)
docs   = [c['content'] for c in chunks]                    # 표시용 원문
titles = [c['metadata'].get('doc_title', '') for c in chunks]
print(f'청크 {len(chunks)}개')
print('예시:', docs[0][:80])

## 2. 임베딩 함수 정의

둘 다 **정규화**해 코사인 유사도를 내적으로 계산할 수 있게 한다.

In [ ]:
def l2_normalize(mat):
    mat = np.asarray(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    return mat / np.clip(norms, 1e-12, None)

# ── OpenAI ──
from openai import OpenAI
_oa = OpenAI()                      # OPENAI_API_KEY 사용
OA_MODEL = 'text-embedding-3-small'

def embed_openai(items, dim=1024, batch=128):
    out = []
    for i in range(0, len(items), batch):
        resp = _oa.embeddings.create(model=OA_MODEL, input=items[i:i+batch], dimensions=dim)
        out.extend(d.embedding for d in resp.data)
    return l2_normalize(out)

# ── KURE-v1 ──
from sentence_transformers import SentenceTransformer
_kure = None
def embed_kure(items, batch=32):
    global _kure
    if _kure is None:
        _kure = SentenceTransformer('nlpai-lab/KURE-v1')
    return _kure.encode(items, batch_size=batch, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True)

## 3. 코퍼스 임베딩 + 시간 측정

In [ ]:
t = time.perf_counter(); emb_oa = embed_openai(texts);  t_oa = time.perf_counter() - t
print(f'OpenAI  : shape {emb_oa.shape}, {t_oa:.1f}s')

t = time.perf_counter(); emb_kure = embed_kure(texts); t_kure = time.perf_counter() - t
print(f'KURE-v1 : shape {emb_kure.shape}, {t_kure:.1f}s')

## 4. 검색 품질 비교 (cosine top-k)

같은 법률 질문을 두 모델로 임베딩해 각자의 코퍼스에서 top-k 를 뽑고 나란히 본다.

In [ ]:
QUERIES = [
    '전세 계약이 끝났는데 집주인이 보증금을 안 돌려줘요',
    '임차권등기명령은 어떻게 신청하나요?',
    '집주인이 수리를 안 해주면 월세를 안 내도 되나요?',
    '계약갱신요구권은 몇 번까지 쓸 수 있나요?',
]
TOPK = 3

def top_k(query_vec, corpus, k=TOPK):
    sims = corpus @ query_vec            # 정규화돼 있으므로 dot == cosine
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

q_oa = embed_openai(QUERIES)
q_kure = embed_kure(QUERIES)

for qi, q in enumerate(QUERIES):
    print('=' * 92)
    print('질문:', q)
    for name, corpus, qv in [('OpenAI', emb_oa, q_oa[qi]), ('KURE-v1', emb_kure, q_kure[qi])]:
        print()
        print(f'[{name}]')
        for rank, (i, s) in enumerate(top_k(qv, corpus), 1):
            preview = ' '.join(docs[i][:70].split())
            print(f'  {rank}. ({s:.3f}) {titles[i][:16]:<16} | {preview}')
    print()

## 5. 요약

처리량과, 두 모델이 뽑은 top-k 결과가 얼마나 겹치는지(Jaccard)를 본다.

In [ ]:
def line(name, emb, secs):
    n = len(emb)
    return f'{name:<9} dim={emb.shape[1]:<5} chunks={n:<5} {secs:6.1f}s  {n/secs:6.1f} chunk/s'

print(line('OpenAI', emb_oa, t_oa))
print(line('KURE-v1', emb_kure, t_kure))

def topk_set(corpus, qv, k=5):
    return {i for i, _ in top_k(qv, corpus, k)}

jac = []
for qi in range(len(QUERIES)):
    a = topk_set(emb_oa, q_oa[qi]); b = topk_set(emb_kure, q_kure[qi])
    jac.append(len(a & b) / len(a | b))

print()
print(f'top-5 결과 겹침(Jaccard) 평균: {np.mean(jac):.2f}  (1.0=완전일치, 0=완전상이)')